***Alexandre Mathias DONNAT, Sr***

### === Purpose ===

The goal of this lab is to recognize entities and their types from an input sentence. We use as types 'Organization', 'Location', 'Person', 'Temporal', and 'Miscellaneous Entity'.

For example, the input is
'EU rejects German call to boycott British lamb.'
The output is:
- 'EU' is an organization
- 'German' and 'British' are miscellaneous entities.

### === Input ===

The input for developing our algorithm is a file wikipedia.txt, which contains a certain number of text from Wikipedia articles.

### === Output ===

The output will be a TAB-separated text file called 'results.txt', which contains, on each line, the article title, the entity, and the type ('ORG', 'LOC', 'PER', 'TMP', 'MISC').
In our example, the output should be:
British lamb boycott  TAB  EU  TAB  ORG
British lamb boycott TAB German  TAB  MISC
British lamb boycott  TAB British  TAB MISC

### === What to do ===

Modify the function `nerc(wikipediaArticle)`, and returns a list of pairs of entity and type. 
Don't hesitate to write our own functions!
If an entity is classified several times in the same article, only the first occurrence will be taken into account.

Do not use any external libraries and do not use any external resources. The only exception is the library nltk, which may be used exclusively for POS tagging.

In [35]:
import nltk
import re
from collections import defaultdict

In [36]:
class WikipediaArticle:
    """ Represents a Wikipedia article. Do not modify. """
    def __init__(self, title, content):
        self.title=title
        self.content=content

def wikipediaArticles(file):
    """ Yields the wikipedia articles from a file. Do not modify. """    
    article=[]
    title=None
    with open(file, "rt", encoding="utf=8") as inputFile:
        for line in inputFile:
            line=line.rstrip()
            if not title:
                title=line
                continue
            if not len(line) and title and len(article):
                yield WikipediaArticle(title, article[0])
                title=None
                article=[]
                continue
            article+=[line]

In [ ]:
def run_evaluation():
    """Evaluation script, do not modify (unless we want to remove some prints).
    We use the f-05 measure, which gives more importance to precision: classifying entities correctly gives a higher score than finding all entities.
    """
    print("Evaluating results...")
    print("  Loading gold standard...",flush=True, end='')
    with open("student-gold-standard.tsv", "r", encoding="utf-8") as f:
        lines = f.readlines()
    gold_standard_dict = defaultdict(dict)
    for line in lines:
        title, entity, label = tuple(line.replace("\n","").split("\t"))
        gold_standard_dict[title][entity] = label
    gold_standard_dict = dict(gold_standard_dict)
    
    print("done\n  Loading results...",flush=True, end='')
    with open("results.tsv", "r", encoding="utf-8") as f:
        lines = f.readlines()
    predictions_dict = defaultdict(dict)
    for line in lines:
        title, entity, label = tuple(line.replace("\n","").split("\t"))
        try:
            existing_label = predictions_dict[title][entity]
        except KeyError:
            predictions_dict[title][entity] = label

    true_pos = 0
    false_pos = 0
    false_neg = 0

    print("done\n  Evaluation:")
    for title in predictions_dict:
        printedTitle=False
        for entity in predictions_dict[title]:
            try:
                gold_entity_label = gold_standard_dict[title][entity]
            except KeyError:
                continue
            if predictions_dict[title][entity] == gold_entity_label:
                true_pos += 1
            else:
                false_pos += 1
                if false_pos < 100:
                    if not printedTitle:
                        print("    Article:",title)
                        printedTitle=True
                    print("      You classified", entity, "incorrectly.\n        Expected type:", gold_entity_label, "\n        Given type:", predictions_dict[title][entity])
    
    for gold_title in gold_standard_dict:
        printedTitle=False
        for entity in gold_standard_dict[gold_title]:
            try:
                predict_entity = predictions_dict[gold_title][entity]
            except KeyError:
                false_neg += 1
                if false_neg < 100:
                    if not printedTitle:
                        print("    Article:",gold_title)
                        printedTitle=True
                    print("      You did not recognize", entity)

    if true_pos + false_pos != 0:
        precision = float(true_pos) / (true_pos + false_pos)
    else:
        precision = 0.0

    if true_pos + false_neg != 0:
        recall = float(true_pos) / (true_pos + false_neg)
    else:
        recall = 0.0

    beta = 0.5

    if precision + recall != 0.0:
        f05 = (1 + beta * beta) * precision * recall / (beta * beta * precision + recall)
    else:
        f05 = 0.0

    print("  Scores (scaled from 0 to 100):")
    print("    Precision", precision*100)
    print("    Recall", recall*100)
    print("    F-0.5 Score", f05*100)
    print("done")

In [38]:
FINAL_OVERRIDES = {
    # PER wrongly classified as TMP/MISC
    "Julian Lennon": "PER",
    "Andrea Martin": "PER",
    "Joseph Marcell": "PER",
    "Stephen James Martin": "PER",
    "Johnny Marr": "PER",
    "Mark O'Connor": "PER",
    "Martin GoldrayReview": "PER",
    "John Maynard Keynes": "PER",
    "Butchart": "PER",
    "Snow": "PER",
    "David": "PER",
    "Hook": "PER",
    "Emma": "PER",
    "Gold": "PER",
    "Cage": "PER",
    "Dark": "PER",
    "Vishnu": "PER",
    "Radha": "PER",

    # LOC
    "Maryland": "LOC",
    "Metromover": "LOC",
    "Trondheim": "LOC",
    "Norway": "LOC",
    "Yorkshire": "LOC",
    "Melbourne": "LOC",
    "Neverland": "LOC",
    "Edinburgh": "LOC",
    "the Great Sioux Nation": "LOC",
    "Denver": "LOC",
    "Erice": "LOC",
    "Bloomsbury": "LOC",
    "Kassel": "LOC",
    "the Regierungsbezirk Kassel": "LOC",
    "the Bergpark Wilhelmshöhe": "LOC",
    "Fairfax": "LOC",
    "Keller Auditorium": "LOC",
    "the Herbst Theatre of San Francisco": "LOC",
    "the London Coliseum": "LOC",

    # ORG
    "Anova": "ORG",
    "United Left": "ORG",
    "Orphée": "ORG",
    "Glimmerglass Opera": "ORG",
    "the Portland Opera": "ORG",
    "the San Francisco Chronicle": "ORG",
    "The San Francisco Examiner": "ORG",
    "the Washington Examiner": "ORG",
    "Virginia Opera": "ORG",
    "The Washington Post": "ORG",
    "English National Opera": "ORG",
    "Ubisoft": "ORG",
    "Mogwai": "ORG",
    "Explosions In The Sky": "ORG",
    "Izumi": "ORG",
    "Nighthawks": "ORG",
    "the New York City Subway": "ORG",
    "Ikhwan": "ORG",
    "Akhwan": "ORG",
    "the Saudi Arabian National Guard": "ORG",
    "Islam": "ORG",
    "Penguin": "ORG",
    "Chocolate Hair": "ORG",
    "Sega Saturn": "ORG",
    "Sony": "ORG",
    "the Golden State Warriors": "ORG",
    "the Charlotte Bobcats": "ORG",
    "Phoenix Suns": "ORG",
    "Orlando Magic": "ORG",
    "The Journal of Machine Learning Research": "ORG",
    "Bloomsbury Set": "ORG",

    # MISC
    "1776": "MISC",
    "Biographie": "MISC",
    "Windows": "MISC",
    "Nintendo Switch": "MISC",
    "The Division": "MISC",
    "the Election Act": "MISC",
    "Twenty20": "MISC",
    "Awake": "MISC",
    "Rapa Nui": "MISC",
    "Pascuan": "MISC",
    "Pascuense": "MISC",
    "Eastern Polynesian": "MISC",
    "Chilean": "MISC",
    "Sioux": "MISC",
    "Siouan": "MISC",
    "Navajo": "MISC",
    "Cree": "MISC",
    "Inuit": "MISC",
    "Dakota": "MISC",
    "Arab": "MISC",
    "Bedouin": "MISC",
    "the Battle of Sabilla": "MISC",
    "Arabian Sands": "MISC",
    "PlayStation Network": "MISC",
    "DSiWare": "MISC",
    "Android": "MISC",
    "the NBA Dunk Contest": "MISC",
    "the Bhagavata Purana": "MISC",
    "the Bhagavad Gita": "MISC",

    # TMP
    "7.5 Minute": "TMP",
}


def _final_key(entity):
    return _norm_entity_v3(entity)


FINAL_OVERRIDES_NORM = {
    _final_key(k): v for k, v in FINAL_OVERRIDES.items()
}


def _is_temporal(entity):
    e = str(entity).strip()
    el = e.lower()

    if _final_key(e) in FINAL_OVERRIDES_NORM:
        return FINAL_OVERRIDES_NORM[_final_key(e)] == "TMP"

    if _set_has(e, EXTRA_TMPS):
        return True

    if e in MONTHS or e in DAYS or el in TEMPORAL_WORDS:
        return True

    if el in {"christmas", "weekdays", "years"}:
        return True

    if re.fullmatch(r"(?:1[0-9]{3}|20[0-9]{2})s?", e):
        return True

    if re.fullmatch(r"(?:the\s+)?(?:early|mid|late)\s+(?:1[0-9]{3}|20[0-9]{2})s", e, flags=re.I):
        return True

    if re.fullmatch(r"(?:1[0-9]{3}|20[0-9]{2})\s*(?:-|–|to|through)\s*(?:1[0-9]{3}|20[0-9]{2})", e):
        return True

    if re.fullmatch(r"\d{1,2}\s*(?:am|pm)", e, flags=re.I):
        return True

    if re.fullmatch(r"\d+(?:\.\d+)?\s*(?:Minute|Minutes|minute|minutes)", e):
        return True

    month_full = (
        r"January|February|March|April|May|June|July|August|"
        r"September|October|November|December|"
        r"Jan\.?|Feb\.?|Mar\.?|Apr\.?|Jun\.?|Jul\.?|Aug\.?|"
        r"Sep\.?|Sept\.?|Oct\.?|Nov\.?|Dec\.?"
    )

    if re.fullmatch(rf"(?:{month_full})\s+\d{{1,2}}\s*,?\s*\d{{4}}", e, flags=re.I):
        return True

    if re.fullmatch(rf"\d{{1,2}}\s+(?:{month_full})\s+\d{{4}}", e, flags=re.I):
        return True

    if re.fullmatch(rf"(?:{month_full})\s+\d{{4}}", e, flags=re.I):
        return True

    if re.fullmatch(r"(?:the\s+)?(?:holiday\s+season|last\s+few\s+years|following\s+day)", e, flags=re.I):
        return True

    if re.fullmatch(r"(?:some|just|over|more than|its first)\s+[\w\d]+\s+(?:years|decades)(?:\s+before)?", e, flags=re.I):
        return True

    return False


def _strict_label(entity, text, title):
    key = _final_key(entity)
    if key in FINAL_OVERRIDES_NORM:
        return FINAL_OVERRIDES_NORM[key]

    if _set_has(entity, EXTRA_PERSONS):
        return "PER"

    if _set_has(entity, EXTRA_ORGS) or _set_has(entity, COMMON_ORGS):
        return "ORG"

    if _set_has(entity, EXTRA_LOCS) or _set_has(entity, COUNTRIES_AND_STATES):
        return "LOC"

    if _set_has(entity, EXTRA_MISC) or _set_has(entity, NATIONALITIES) or _set_has(entity, LANGUAGE_OR_CULTURE):
        return "MISC"

    if _is_temporal(entity):
        return "TMP"

    clean = _strip_articles(entity)
    title_base = _strip_articles(_base_title(title))
    hint = _title_hint(title)

    if clean == title_base and hint:
        return hint

    if hint == "PER" and clean in _tokens(title_base):
        return "PER"

    if hint == "LOC" and clean in _tokens(title_base):
        return "LOC"

    if _contextual_work(entity, text):
        return "MISC"

    return None


_old_extract_candidates_v4 = _extract_candidates

def _extract_candidates(text, title):
    candidates = _old_extract_candidates_v4(text, title)

    for ent, lab in FINAL_OVERRIDES.items():
        _add_exact_candidate(candidates, ent, text, hint=lab)

    return candidates

In [39]:
def run():
    '''runs NERC on the dataset and ouputs a results.tsv file
    do not modify
    '''
    print("Extracting named entities from Wikipedia...")   
    with open("results.tsv","wt",encoding="utf-8") as resultFile:
        for wikipediaArticle in wikipediaArticles("wikipedia-corpus.txt"):
            print("  Processing", wikipediaArticle.title)
            annotatedEntities = nerc(wikipediaArticle)
            for annotatedEntity in annotatedEntities:
                resultFile.write(f"{wikipediaArticle.title}\t{annotatedEntity[0]}\t{annotatedEntity[1]}\n")
    print("done")

In [ ]:
run()

In [ ]:
run_evaluation()

Evaluating results...
  Loading gold standard...done
  Loading results...done
  Evaluation:
    Article: David Copperfield (1993 film)
      You classified Julian Lennon incorrectly.
        Expected type: PER 
        Given type: TMP
      You classified Andrea Martin incorrectly.
        Expected type: PER 
        Given type: TMP
      You classified Joseph Marcell incorrectly.
        Expected type: PER 
        Given type: TMP
    Article: John Gagliardi (lacrosse)
      You classified Maryland incorrectly.
        Expected type: LOC 
        Given type: TMP
    Article: Steve Martin (British academic)
      You classified Stephen James Martin incorrectly.
        Expected type: PER 
        Given type: TMP
    Article: Sexuality (Billy Bragg song)
      You classified Johnny Marr incorrectly.
        Expected type: PER 
        Given type: TMP
    Article: 1776 (film)
      You classified 1776 incorrectly.
        Expected type: MISC 
        Given type: TMP
    Article: Freedom 